
{% include nav/toolkits/bathroom/menu.html %}
    
    <link href="https://fonts.googleapis.com/icon?family=Material+Icons" rel="stylesheet">


<style>
        .container {
            position: relative;
            top: 10;
            margin-top: 10 !important;
        }
        #additionalInput {
            display: none; /* Initially hidden */
        }
        body {
            color: black; 
        }
            /* Header Styling */
    .header-container {
        text-align: center;
        margin-top: 20px;
    }

    .header-container h1 {
        font-size: 28px;
        color: white;
    }
    </style>


<script type="module">
    import { javaURI, pythonURI, fetchOptions } from '{{ site.baseurl }}/assets/js/api/config.js';
    document.addEventListener("DOMContentLoaded", function () {
            // Add click event listener to the "Request Hall Pass" button
            document.getElementById("requestPassBtn").addEventListener("click", requestHallPass);
            document.getElementById("checkinBtn").addEventListener("click", checkinHallPass);
        });
   async function fetchTeacherData() {
    const storedTeacherFirstName = localStorage.getItem("teacherFirstName");
    const storedTeacherLastName = localStorage.getItem("teacherLastName");

    if (!storedTeacherFirstName || !storedTeacherLastName) {
        document.getElementById("teacher-form").style.display = "block"; // Show the form to enter teacher name
        return;
    }

    try {
        let response = await fetch(`${javaURI}/api/hallpass/getTeacher?fname=${storedTeacherFirstName}&lname=${storedTeacherLastName}`, fetchOptions);
        if (!response.ok) throw new Error("Network response was not ok");

        const contentType = response.headers.get("content-type");
        let data = contentType && contentType.includes("application/json") ? await response.json() : await response.text();

        if (!data || (typeof data === "object" && Object.keys(data).length === 0)) {
            document.getElementById("error-message").textContent = "Teacher not found. Please enter your teacher’s name.";
            document.getElementById("error-container").style.display = "block";
            document.getElementById("teacher-form").style.display = "block";
        } else {
            document.getElementById("active-teacher").textContent = `${storedTeacherFirstName} ${storedTeacherLastName}`;
            await fetchActivePass();
        }
    } catch (error) {
        console.error("Error fetching teacher data:", error);
        document.getElementById("error-message").textContent = "An error occurred while fetching data. Please try again later.";
        document.getElementById("error-container").style.display = "block";
    }
}


    async function fetchActivePass() {
    const storedTeacherFirstName = localStorage.getItem("teacherFirstName");
    const storedTeacherLastName = localStorage.getItem("teacherLastName");

    try {
        let response = await fetch(`${javaURI}/api/hallpass/getactivepass?email=toby`, fetchOptions);
        if (!response.ok) throw new Error("Network response was not ok");

        const contentType = response.headers.get("content-type");
        let data = contentType && contentType.includes("application/json") ? await response.json() : await response.text();

        if (!data || (typeof data === "object" && Object.keys(data).length === 0)) {
            document.getElementById("pass-form").style.display = "block";
        } else {
            document.getElementById("active-pass-container").style.display = "block";
            document.getElementById("active-teacher").textContent = `${storedTeacherFirstName} ${storedTeacherLastName}`;
            document.getElementById("active-activity").textContent = data.activity || "Not Available";
            document.getElementById("active-period").textContent = data.period || "Not Available";
        }
    } catch (error) {
        console.error("Error fetching active pass:", error);
        document.getElementById("error-message").textContent = "An error occurred while fetching data. Please try again later.";
        document.getElementById("error-container").style.display = "block";
    }
}

async function requestHallPass() {
    const teacherFirstName = document.getElementById("teacherFirstName").value.trim();
    const teacherLastName = document.getElementById("teacherLastName").value.trim();
    const period = document.getElementById("period").value;
    const activity = document.getElementById("activity").value;
    const resultDiv = document.getElementById("confirm-message");

    // Save the teacher name in local storage
    localStorage.setItem("teacherFirstName", teacherFirstName);
    localStorage.setItem("teacherLastName", teacherLastName);

    resultDiv.innerHTML = ""; // Clear previous results

    const requestBody = {
        userName: "toby",
        teacherFirstName: teacherFirstName,
        teacherLastName: teacherLastName,
        period: period,
        activity: activity
    };

    try {
        const response = await fetch(`${javaURI}/api/hallpass/request`, {
            method: "POST",
            headers: {
                "Content-Type": "application/json"
            },
            body: JSON.stringify(requestBody)
        });

        if (!response.ok) {
            throw new Error(`HTTP error! Status: ${response.status}`);
        }

        document.getElementById("pass-form").style.display = "none";
        document.getElementById("active-pass-container").style.display = "block";
        resultDiv.innerHTML = `<span style="color: green;font-size: 20px;">Success! Your hall pass has been processed.</span>`;
    } catch (error) {
        console.error("Error requesting hall pass:", error);
    }
}
    
    async function checkinHallPass() {

        try {
            const response = await fetch(javaURI + "/api/hallpass/checkout?email=toby", {
                method: "POST",
                headers: {
                    "Content-Type": "application/json"
                }
            });
            const resultDiv = document.getElementById("confirm-message");

            resultDiv.innerHTML = ""; // Clear previous results

            if (!response.ok) {
                throw new Error(`HTTP error! Status: ${response.status}`);
            }

            const data = await response.text();

            document.getElementById("pass-form").style.display = "none";
            document.getElementById("active-pass-container").style.display = "none";
            document.getElementById("confirmation-container").style.display = "block";
            
        resultDiv.innerHTML = `<span style="color: green;font-size: 20px;">Success: Thank You! Your hall pass visit has been recorded. Have a nice day!.</span>`; 
        } catch (error) {
            //resultDiv.innerHTML = `<span style="color: red;">${error.message}</span>`;
        }
    }

    fetchTeacherData();
</script>

<div class="header-container">
    <h1 style="color:#0947ab;">Welcome to Hall Pass Request</h1>
    <img src="https://github.com/user-attachments/assets/0f91ecf3-72a3-4712-b0ba-8d0f8ca55bb3">
</div>
    <div class="container bg-primary text-black">
    <div id = "confirmation-container" class="bg-light rounded" style="display: none;">
        <p id="confirm-message"></p>
    </div>
        <div id="error-container" class="alert alert-danger" role="alert" style="display: none;">
            <p id="error-message"></p>
        </div> 
        <div id="active-pass-container" class="bg-light rounded" style="display: none;">
            <h2>You have an active hall pass. Please check-in to end session</h2>
            <p><strong>Teacher:</strong> <span id="active-teacher"></span></p>
            <p><strong>Activity:</strong> <span id="active-activity"></span></p>
            <p><strong>Period:</strong> <span id="active-period"></span></p>
                <button id="checkinBtn" class="btn btn-primary mt-3">Check-in Hall Pass</button>            
        </div>
       <div id="teacher-form" class="bg-light rounded" style="display: none;">
    <h2>Enter Information</h2>
    <div class="form-group">
        <label for="teacherFirstName">Teacher First Name:</label>
        <input type="text" id="teacherFirstName" name="teacherFirstName" class="form-control" required>
    </div>
    <div class="form-group">
        <label for="teacherLastName">Teacher Last Name:</label>
        <input type="text" id="teacherLastName" name="teacherLastName" class="form-control" required>
    </div>
    <button type="button" class="btn btn-primary mt-2" onclick="saveTeacher()">Submit</button>
</div>
<button id="requestPassBtn" class="btn btn-primary mt-3">Request Hall Pass</button>         
        </div>
    </div>

 
  


 